In [ ]:
from dataclasses import dataclass
from abc import ABC

class Tokenizer(ABC):
    """Abstract interface for a tokenizer."""
    def encode(self, string: str) -> list[int]:
        raise NotImplementedError
    def decode(self, indices: list[int]) -> str:
        raise NotImplementedError


@dataclass(frozen=True)
class BPETokenizerParams:
    """
    All you need to specify a BPETokenizer.

    예시:
        vocab = {
            0: b"a",
            1: b"b",
            2: b"c",
            3: b"ab",  # 추가로 병합된 토큰
        }
        merges = {
            (0, 1): 3,  # 토큰 0('a')과 1('b')를 병합하여 3('ab') 생성
        }
        params = BPETokenizerParams(vocab=vocab, merges=merges)
    """
    vocab: dict[int, bytes]     # index -> bytes
    merges: dict[tuple[int, int], int]  # index1,index2 -> new_index
    
class CharacterTokenizer(Tokenizer):
    """Represent a string as a sequence of Unicode code points."""
    def encode(self, string: str) -> list[int]:
        return list(map(ord, string))
    def decode(self, indices: list[int]) -> str:
        return "".join(map(chr, indices))
    
class ByteTokenizer(Tokenizer):
    """
    Represent a string as a sequence of bytes.

    영어 설명:
        Example:
            >>> tokenizer = ByteTokenizer()
            >>> indices = tokenizer.encode("Hello, world!")
            >>> print(indices)
            [72, 101, 108, 108, 111, 44, 32, 119, 111, 114, 108, 100, 33]
            >>> decoded = tokenizer.decode(indices)
            >>> print(decoded)
            Hello, world!

        Decoding explanation:
            Decoding just reconstructs the original string from the byte indices.
            We first convert the list of integer byte values into a bytes object
            (using Python's built-in `bytes()` constructor). This is exactly the
            byte-for-byte result of UTF-8 encoding. 

            Then, `.decode("utf-8")` takes those bytes and returns the original string.

            Note: The bytes in UTF-8 do *not* correspond 1-to-1 with Unicode code points. 
            For many non-ASCII characters, one code point (a character) consists of several bytes.
            For example:
                indices = [236, 176, 180]  # UTF-8 bytes for "반" (Unicode code point U+BC18)
                string_bytes = bytes(indices)
                string = string_bytes.decode("utf-8")  # returns "반"
            In this case, 3 bytes → 1 character.

            In short: 
            - Byte-level tokenization operates on UTF-8 bytes, not code points.
            - Decoding bytes with UTF-8 will reassemble the original string, even though
              the number of bytes may differ from the number of code points.
            - There is no direct mapping between a single UTF-8 byte and a single code point, except for ASCII (U+0000 to U+007F).

    한글 설명:
        문자열을 UTF-8 바이트 시퀀스로 토크나이즈(분해)하는 방법입니다.
        
        예시:
        >>> tokenizer = ByteTokenizer()
        >>> indices = tokenizer.encode("반가워!")
        >>> print(indices)
        [236, 176, 180, 234, 176, 128, 236, 149, 136, 33]  # "반", "가", "워", "!"의 UTF-8 바이트 시퀀스

        - 한글(또는 기타 유니코드 문자)은 한 글자(코드포인트)가 여러 개의 바이트로 인코딩됩니다.
        - decode에서는 이 바이트 리스트를 bytes로 만들고, 다시 .decode("utf-8")을 통해 원래 문자열로 복원합니다.
        - 한글처럼 비ASCII 문자는 1글자 = 2~4 바이트(UTF-8)에 해당할 수 있습니다.
        - 즉, 바이트 토크나이저는 문자열을 코드포인트(문자 단위)가 아니라 UTF-8 바이트 단위로 나누고, 복원은 항상 안전합니다.

        참고: ASCII 문자(U+0000~U+007F)는 1글자=1바이트이므로 1:1 매핑이지만,
        그 외 한글, 이모지 등은 여러 바이트로 표현됩니다.

    """
    def encode(self, string: str) -> list[int]:
        string_bytes = string.encode("utf-8")
        indices = list(map(int, string_bytes))
        print(f"string_bytes: {string_bytes}, indices: {indices}")
        return indices
    def decode(self, indices: list[int]) -> str:
        # bytes(indices)로 정수 리스트를 바이트로 만들고 utf-8로 디코딩(복원)합니다.
        string_bytes = bytes(indices)
        string = string_bytes.decode("utf-8")
        return string

In [22]:
idices = ByteTokenizer().encode("반가워! Hello, world! 🎉")

string_bytes: b'\xeb\xb0\x98\xea\xb0\x80\xec\x9b\x8c! Hello, world! \xf0\x9f\x8e\x89', indices: [235, 176, 152, 234, 176, 128, 236, 155, 140, 33, 32, 72, 101, 108, 108, 111, 44, 32, 119, 111, 114, 108, 100, 33, 32, 240, 159, 142, 137]


In [ ]:
def merge(indices: list[int], pair: tuple[int, int], new_index: int) -> list[int]:  # @inspect indices, @inspect pair, @inspect new_index
    """Return `indices`, but with all instances of `pair` replaced with `new_index`."""
    new_indices = []  # @inspect new_indices
    i = 0  # @inspect i
    while i < len(indices):
        if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
            new_indices.append(new_index)
            i += 2
        else:
            new_indices.append(indices[i])
            i += 1
    print(f"new_indices: {new_indices}")
    return new_indices

class BPETokenizer(Tokenizer):
    """BPE tokenizer given a set of merges and a vocabulary."""
    def __init__(self, params: BPETokenizerParams):
        self.params = params
    def encode(self, string: str) -> list[int]:
        indices = list(map(int, string.encode("utf-8")))  # @inspect indices
        # Note: this is a very slow implementation
        for pair, new_index in self.params.merges.items():  # @inspect pair, @inspect new_index
            indices = merge(indices, pair, new_index)
        return indices
    def decode(self, indices: list[int]) -> str:
        bytes_list = list(map(self.params.vocab.get, indices))  # @inspect bytes_list
        string = b"".join(bytes_list).decode("utf-8")  # @inspect string
        return string
    
def get_compression_ratio(string: str, indices: list[int]) -> float:
    """
    압축률(Compression Ratio)을 구하는 함수입니다.
    원본 문자열 string과 토크나이즈된 인덱스 리스트 indices를 받아,
    원본 문자열의 바이트 수를 토큰 수로 나눈 값을 반환합니다.
    (즉, 1 token당 평균 몇 byte가 압축되었는지를 나타냄)
    """
    num_bytes = len(string.encode("utf-8"))    # 원본 문자열의 바이트 길이
    num_tokens = len(indices)                  # 토큰의 개수
    if num_tokens == 0:
        return 0.0
    return num_bytes / num_tokens

In [43]:
!uv pip install tiktoken

Using Python 3.12.10 environment at: /Users/luke/workspace/CS336-Language-Modeling-from-Scratch/.venv
Resolved 7 packages in 270ms                                         
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)----     0 B/282.42 KiB                    
⠙ Preparing packages... (0/2)---- 16.00 KiB/282.42 KiB                  
⠙ Preparing packages... (0/2)---- 32.00 KiB/282.42 KiB                  
⠙ Preparing packages... (0/2)---- 48.00 KiB/282.42 KiB                  
⠙ Preparing packages... (0/2)---- 64.00 KiB/282.42 KiB                  
⠙ Preparing packages... (0/2)---- 80.00 KiB/282.42 KiB                  
⠙ Preparing packages... (0/2)---- 96.00 KiB/282.42 KiB                  
⠙ Preparing packages... (0/2)---- 112.00 KiB/282.42 KiB                 
⠙ Preparing packages... (0/2)---- 128.00 KiB/282.42 KiB                 
⠙ Preparing packages... (0/2)---- 144.00 KiB/282.42 KiB                 
⠙ Preparing packa

In [44]:
import tiktoken
def get_gpt2_tokenizer():
    # Code: https://github.com/openai/tiktoken
    # You can use cl100k_base for the gpt3.5-turbo or gpt4 tokenizer
    return tiktoken.get_encoding("gpt2")

In [47]:
def tokenization_examples():
    # Here's the GPT-2 tokenizer from OpenAI (tiktoken) in action.
    tokenizer = get_gpt2_tokenizer()
    string = "Hello, 🌍! 언어모델!"  # @inspect string
    # Check that encode() and decode() roundtrip:
    indices = tokenizer.encode(string)  # @inspect indices
    reconstructed_string = tokenizer.decode(indices)  # @inspect reconstructed_string
    assert string == reconstructed_string
    compression_ratio = get_compression_ratio(string, indices)  # @inspect compression_ratio
    print(f"string: {string}, indices: {indices}, reconstructed_string: {reconstructed_string}, compression_ratio: {compression_ratio}")    
tokenization_examples()

string: Hello, 🌍! 언어모델!, indices: [15496, 11, 12520, 234, 235, 0, 23821, 244, 116, 168, 244, 112, 167, 103, 101, 167, 235, 116, 0], reconstructed_string: Hello, 🌍! 언어모델!, compression_ratio: 1.368421052631579
